## Load Data dari Kaggle

In [0]:
# Cell 1 - Install kaggle
%pip install kaggle

In [0]:
# Cell 2 - Setup credentials
import os

# Isi dengan isi dari kaggle.json kamu
os.environ['KAGGLE_USERNAME'] = '' # Pakai username Kaggle sendiri
os.environ['KAGGLE_KEY'] = ''  # Pakai API Kaggle sendiri (legacy API)

In [0]:
# Cell 3 - Download dataset
import subprocess
import os

# Download to /tmp (larger storage)
download_path = "/Volumes/workspace/default/nyc_taxi_volume"
os.chdir(download_path)

# Download train.csv file
subprocess.run([
    "kaggle", "competitions", "download",
    "-c", "new-york-city-taxi-fare-prediction",
    "-f", "train.csv"
], check=True)

print(f"Download selesai! File tersimpan di {download_path}")

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

# Cek file hasil download
for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

In [0]:
# Cell 5 — Unzip langsung di dalam Volume
import zipfile

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

zip_path   = f"{VOLUME_PATH}/train.csv.zip"
extract_to = VOLUME_PATH

print("Mulai unzip...")
with zipfile.ZipFile(zip_path, 'r') as z:
    members = z.namelist()
    print(f"File di dalam zip: {members}")
    z.extractall(extract_to)
    print(f"Extracted ke: {extract_to}")

# Verifikasi hasil unzip
print("\nIsi Volume setelah unzip:")
for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

##  Setup & Load Data

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName("NYC_Taxi_Full_Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

# Helper: simpan ke Parquet lalu baca ulang (pengganti .cache()/.persist())
def checkpoint(df, name):
    path = f"{VOLUME_PATH}/checkpoints/{name}"
    df.write.mode("overwrite").parquet(path)
    df_loaded = spark.read.parquet(path)
    print(f"Checkpoint '{name}' disimpan & dimuat ulang.")
    return df_loaded

schema = StructType([
    StructField("key",               StringType(),  True),
    StructField("fare_amount",       DoubleType(),  True),
    StructField("pickup_datetime",   StringType(),  True),
    StructField("pickup_longitude",  DoubleType(),  True),
    StructField("pickup_latitude",   DoubleType(),  True),
    StructField("dropoff_longitude", DoubleType(),  True),
    StructField("dropoff_latitude",  DoubleType(),  True),
    StructField("passenger_count",   IntegerType(), True),
])

print("Loading train.csv...")
df_raw = spark.read.csv(
    f"{VOLUME_PATH}/train.csv",
    header=True,
    schema=schema
)

# Checkpoint pengganti .cache()
df_raw     = checkpoint(df_raw, "raw")
total_rows = df_raw.count()

print(f"Total baris : {total_rows:,}")
print(f"Total kolom : {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
print("=" * 60)
print("SAMPLE DATA (10 baris pertama)")
print("=" * 60)
display(df_raw.limit(10))

print("=" * 60)
print("STATISTIK DESKRIPTIF")
print("=" * 60)
display(df_raw.describe())

## EDA: Analisis Missing Values & Duplikasi

In [0]:
print("=" * 60)
print("ANALISIS MISSING VALUES")
print("=" * 60)

from pyspark.sql.types import NumericType

# Pisahkan kolom numerik dan non-numerik
numeric_cols    = [f.name for f in df_raw.schema.fields if isinstance(f.dataType, NumericType)]
non_numeric_cols = [f.name for f in df_raw.schema.fields if not isinstance(f.dataType, NumericType)]

print(f"Kolom numerik     : {numeric_cols}")
print(f"Kolom non-numerik : {non_numeric_cols}")

# Cek missing: isnan hanya untuk numerik, isNull untuk semua
missing_exprs = []

for c in df_raw.columns:
    if c in numeric_cols:
        # Numerik: cek isNull DAN isnan (NaN adalah nilai float khusus)
        missing_exprs.append(
            spark_sum(
                when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)
            ).alias(c)
        )
    else:
        # String/non-numerik: cek isNull dan string kosong saja
        missing_exprs.append(
            spark_sum(
                when(col(c).isNull() | (trim(col(c)) == ""), 1).otherwise(0)
            ).alias(c)
        )

missing_df = df_raw.select(missing_exprs).toPandas().T.rename(columns={0: "missing_count"})
missing_df["missing_pct"] = (missing_df["missing_count"] / total_rows * 100).round(4)

print("\nSemua kolom (termasuk yang tidak ada missing):")
print(missing_df.to_string())

missing_only = missing_df[missing_df["missing_count"] > 0]
print("\nKolom dengan missing values:")
if missing_only.empty:
    print("  Tidak ada missing values!")
else:
    print(missing_only.to_string())

# Duplikasi — key adalah string jadi langsung .distinct() tanpa isnan
total_unique_keys = df_raw.select("key").distinct().count()
duplikat          = total_rows - total_unique_keys
print(f"\nTotal baris         : {total_rows:,}")
print(f"Unique keys         : {total_unique_keys:,}")
print(f"Duplikasi           : {duplikat:,} baris ({duplikat/total_rows*100:.2f}%)")

## EDA: Distribusi Fare Amount

In [0]:
# Sample kecil untuk visualisasi (tidak ubah df utama)
pdf_fare = df_raw.select("fare_amount") \
    .filter(col("fare_amount").between(0, 200)) \
    .sample(fraction=0.005, seed=42) \
    .toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(pdf_fare["fare_amount"], bins=80, color="#5B8FF9", edgecolor="white", linewidth=0.3)
axes[0].set_title("Distribusi Fare Amount", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Fare (USD)")
axes[0].set_ylabel("Frekuensi")
axes[0].axvline(pdf_fare["fare_amount"].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: ${pdf_fare["fare_amount"].mean():.2f}')
axes[0].axvline(pdf_fare["fare_amount"].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: ${pdf_fare["fare_amount"].median():.2f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(pdf_fare["fare_amount"], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#5B8FF9', alpha=0.7))
axes[1].set_title("Boxplot Fare Amount", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Fare (USD)")

plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eda_fare_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan ke Volume.")

## EDA: Distribusi Passenger Count & Koordinat

In [0]:
pdf_pass = df_raw.groupBy("passenger_count").count() \
    .orderBy("passenger_count").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart passenger
colors = ['#5B8FF9','#61DDAA','#F6BD16','#E86452','#9661BC','#6DC8EC','#FF6D87']
bars = axes[0].bar(
    pdf_pass["passenger_count"].astype(str),
    pdf_pass["count"],
    color=colors[:len(pdf_pass)],
    edgecolor='white'
)
axes[0].set_title("Distribusi Jumlah Penumpang", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Jumlah Penumpang")
axes[0].set_ylabel("Jumlah Trip")
for bar, val in zip(bars, pdf_pass["count"]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5000,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# Scatter pickup location (NYC area)
pdf_coords = df_raw.select("pickup_longitude", "pickup_latitude") \
    .filter(
        col("pickup_longitude").between(-74.05, -73.75) &
        col("pickup_latitude").between(40.60, 40.90)
    ).sample(fraction=0.002, seed=42).toPandas()

axes[1].scatter(
    pdf_coords["pickup_longitude"],
    pdf_coords["pickup_latitude"],
    s=0.3, alpha=0.4, color="#5B8FF9"
)
axes[1].set_title("Distribusi Lokasi Pickup (NYC)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eda_passenger_coords.png", dpi=150, bbox_inches='tight')
plt.show()

## EDA: Analisis Temporal

In [0]:
# ── Ekstrak fitur waktu dari raw data ──────────────────────────
# Fix: buang suffix " UTC" sebelum parsing timestamp
df_time = df_raw \
    .withColumn(
        "pickup_dt",
        to_timestamp(
            regexp_replace(col("pickup_datetime"), " UTC$", ""),
            "yyyy-MM-dd HH:mm:ss"
        )
    ) \
    .withColumn("hour",        hour("pickup_dt")) \
    .withColumn("day_of_week", dayofweek("pickup_dt")) \
    .withColumn("month",       month("pickup_dt")) \
    .withColumn("year",        year("pickup_dt"))

# Verifikasi hasil parsing
print("Sample hasil parsing timestamp:")
df_time.select("pickup_datetime", "pickup_dt", "hour", "day_of_week", "month", "year") \
       .filter(col("pickup_dt").isNotNull()) \
       .limit(5) \
       .show(truncate=False)

null_dt_count = df_time.filter(col("pickup_dt").isNull()).count()
print(f"Baris gagal parse (pickup_dt = null): {null_dt_count:,}")

In [0]:
# ── Agregasi ───────────────────────────────────────────────────
hourly = df_time.groupBy("hour").agg(
    avg("fare_amount").alias("avg_fare"),
    count("*").alias("trip_count")
).orderBy("hour").toPandas()

daily = df_time.groupBy("day_of_week").agg(
    avg("fare_amount").alias("avg_fare"),
    count("*").alias("trip_count")
).orderBy("day_of_week").toPandas()
day_labels = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

yearly = df_time.groupBy("year").agg(
    avg("fare_amount").alias("avg_fare"),
    count("*").alias("trip_count")
).orderBy("year").toPandas()

# ── Visualisasi ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Per jam
ax  = axes[0]
ax2 = ax.twinx()
ax.bar(hourly["hour"], hourly["trip_count"], color="#5B8FF9", alpha=0.4, label="Trip count")
ax2.plot(hourly["hour"], hourly["avg_fare"], color="#E86452", marker='o', markersize=4, linewidth=2, label="Avg fare")
ax.set_title("Trip & Tarif per Jam", fontsize=12, fontweight='bold')
ax.set_xlabel("Jam")
ax.set_ylabel("Jumlah Trip", color="#5B8FF9")
ax2.set_ylabel("Rata-rata Tarif (USD)", color="#E86452")

# Per hari
ax  = axes[1]
ax2 = ax.twinx()
ax.bar(range(len(daily)), daily["trip_count"], color="#61DDAA", alpha=0.4)
ax2.plot(range(len(daily)), daily["avg_fare"], color="#E86452", marker='o', markersize=4, linewidth=2)
ax.set_xticks(range(len(daily)))
ax.set_xticklabels([day_labels[int(d) - 1] for d in daily["day_of_week"]])
ax.set_title("Trip & Tarif per Hari", fontsize=12, fontweight='bold')
ax.set_xlabel("Hari")
ax.set_ylabel("Jumlah Trip", color="#61DDAA")
ax2.set_ylabel("Rata-rata Tarif (USD)", color="#E86452")

# Per tahun
ax  = axes[2]
ax2 = ax.twinx()
ax.bar(yearly["year"].astype(str), yearly["trip_count"], color="#F6BD16", alpha=0.4)
ax2.plot(range(len(yearly)), yearly["avg_fare"], color="#E86452", marker='o', markersize=4, linewidth=2)
ax.set_title("Trip & Tarif per Tahun", fontsize=12, fontweight='bold')
ax.set_xlabel("Tahun")
ax.set_ylabel("Jumlah Trip", color="#F6BD16")
ax2.set_ylabel("Rata-rata Tarif (USD)", color="#E86452")

plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eda_temporal.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan.")

## EDA: Korelasi Fitur

In [0]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

# Gunakan df_time yang sudah ada fitur temporal
df_corr_input = df_time.select(
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count", "hour"
).dropna()

assembler_corr = VectorAssembler(
    inputCols=["fare_amount", "pickup_longitude", "pickup_latitude",
               "dropoff_longitude", "dropoff_latitude", "passenger_count", "hour"],
    outputCol="corr_features"
)
df_corr_vec = assembler_corr.transform(df_corr_input)
corr_matrix = Correlation.corr(df_corr_vec, "corr_features").collect()[0][0].toArray()

labels = ["fare", "pickup_lon", "pickup_lat", "dropoff_lon", "dropoff_lat", "passengers", "hour"]

plt.figure(figsize=(9, 7))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f",
    xticklabels=labels, yticklabels=labels,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True
)
plt.title("Correlation Matrix — Fitur Numerik", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eda_correlation.png", dpi=150, bbox_inches='tight')
plt.show()

## PREPROCESSING: Cleaning & Feature Engineering

In [0]:
print("=" * 60)
print("PREPROCESSING")
print("=" * 60)

df_parsed = df_raw \
    .withColumn(
        "pickup_dt",
        to_timestamp(
            regexp_replace(col("pickup_datetime"), " UTC$", ""),
            "yyyy-MM-dd HH:mm:ss"
        )
    ) \
    .withColumn("hour",        hour("pickup_dt")) \
    .withColumn("day_of_week", dayofweek("pickup_dt")) \
    .withColumn("month",       month("pickup_dt")) \
    .withColumn("year",        year("pickup_dt")) \
    .withColumn("is_weekend",
        when(dayofweek("pickup_dt").isin([1, 7]), 1).otherwise(0)
    )

df_filtered = df_parsed.filter(
    col("fare_amount").between(2.5, 500)             &
    col("pickup_longitude").between(-74.05, -73.75)  &
    col("pickup_latitude").between(40.60, 40.90)     &
    col("dropoff_longitude").between(-74.05, -73.75) &
    col("dropoff_latitude").between(40.60, 40.90)    &
    col("passenger_count").between(1, 6)             &
    col("pickup_dt").isNotNull()
)

# Checkpoint setelah filtering
df_filtered = checkpoint(df_filtered, "filtered")

count_filtered = df_filtered.count()
print(f"Baris sebelum filter : {total_rows:,}")
print(f"Baris setelah filter : {count_filtered:,}")
print(f"Baris dihapus        : {total_rows - count_filtered:,}")

## PREPROCESSING: Haversine Distance & Fitur Lanjutan

In [0]:
def add_haversine(df):
    R = 6371.0
    df = df \
        .withColumn("dlat", radians(col("dropoff_latitude")  - col("pickup_latitude"))) \
        .withColumn("dlon", radians(col("dropoff_longitude") - col("pickup_longitude"))) \
        .withColumn("a",
            sin(col("dlat") / 2) ** 2 +
            cos(radians(col("pickup_latitude"))) *
            cos(radians(col("dropoff_latitude"))) *
            sin(col("dlon") / 2) ** 2
        ) \
        .withColumn("distance_km", lit(2.0 * R) * asin(sqrt(col("a")))) \
        .drop("dlat", "dlon", "a")
    return df

df_feat = add_haversine(df_filtered)

df_feat = df_feat \
    .withColumn("abs_lat_diff", abs(col("dropoff_latitude")  - col("pickup_latitude"))) \
    .withColumn("abs_lon_diff", abs(col("dropoff_longitude") - col("pickup_longitude"))) \
    .withColumn("is_rush_hour",
        when(
            ((col("hour") >= 7)  & (col("hour") <= 9)) |
            ((col("hour") >= 17) & (col("hour") <= 19)), 1
        ).otherwise(0)
    ) \
    .withColumn("is_night",
        when((col("hour") >= 22) | (col("hour") <= 5), 1).otherwise(0)
    )

FEATURE_COLS = [
    "distance_km", "abs_lat_diff", "abs_lon_diff",
    "hour", "day_of_week", "month", "year",
    "passenger_count", "is_weekend", "is_rush_hour", "is_night"
]
TARGET_COL = "fare_amount"

df_model = df_feat.select(FEATURE_COLS + [TARGET_COL]).dropna()

# Checkpoint pengganti .cache()
df_model = checkpoint(df_model, "model_ready")

print(f"Data siap model: {df_model.count():,} baris | {len(FEATURE_COLS)} fitur")
display(df_model.describe())

## PREPROCESSING: Assemble + Scale + Split

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# Hapus StandardScaler — tidak dibutuhkan untuk RF dan GBT
# Tree-based models tidak sensitif terhadap skala fitur
# StandardScaler hanya wajib untuk Linear Regression / SVM / KNN

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",       # langsung "features", tidak perlu "features_raw"
    handleInvalid="skip"
)

# Fit dan transform
df_scaled = assembler.transform(df_model).select("features", TARGET_COL)

# Checkpoint sebelum split
df_scaled = checkpoint(df_scaled, "scaled")

# Split
train_df, val_df, test_df = df_scaled.randomSplit([0.70, 0.15, 0.15], seed=42)

train_df = checkpoint(train_df, "split_train")
val_df   = checkpoint(val_df,   "split_val")
test_df  = checkpoint(test_df,  "split_test")

print(f"Train : {train_df.count():,}")
print(f"Val   : {val_df.count():,}")
print(f"Test  : {test_df.count():,}")

## MODELING: Linear Regression

In [0]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

eval_rmse = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

def evaluate(preds, model_name):
    rmse = eval_rmse.evaluate(preds)
    mae  = eval_mae.evaluate(preds)
    r2   = eval_r2.evaluate(preds)
    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    return {"model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2}

# ── Linear Regression tanpa StandardScaler ──────────────────────
# Kompensasi: naikkan maxIter dan turunkan regParam agar konvergen
# tanpa scaling. Hasilnya tetap valid untuk perbandingan model.
print("Training Linear Regression (tanpa scaler)...")

lr = LinearRegression(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxIter=100,          # lebih banyak iterasi untuk kompensasi tanpa scaling
    regParam=0.01,        # regularisasi lebih kecil
    elasticNetParam=0.0,  # pure Ridge agar lebih stabil
    standardization=True  # LR punya built-in standardization internal — gunakan ini
)

lr_model  = lr.fit(train_df)
lr_preds  = lr_model.transform(test_df)
lr_result = evaluate(lr_preds, "Linear Regression")

print(f"\n  Training summary:")
print(f"  Iterasi           : {lr_model.summary.totalIterations}")
print(f"  Loss awal         : {lr_model.summary.objectiveHistory[0]:.6f}")
print(f"  Loss akhir        : {lr_model.summary.objectiveHistory[-1]:.6f}")

print(f"\n  Koefisien per fitur:")
for name, coef in zip(FEATURE_COLS, lr_model.coefficients):
    print(f"    {name:<20}: {coef:+.6f}")
print(f"  Intercept         : {lr_model.intercept:+.6f}")

## MODELING: Random Forest

In [0]:
# ── Random Forest — tidak perlu scaling ──
print("Training Random Forest...")
rf = RandomForestRegressor(
    featuresCol="features",      # langsung dari VectorAssembler
    labelCol=TARGET_COL,
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=5,
    featureSubsetStrategy="auto",
    seed=42
)
rf_model  = rf.fit(train_df)
rf_preds  = rf_model.transform(test_df)
rf_result = evaluate(rf_preds, "Random Forest (100 trees, depth 10)")

print("\n  Feature Importances:")
importances = sorted(
    zip(FEATURE_COLS, rf_model.featureImportances),
    key=lambda x: x[1], reverse=True
)
for name, imp in importances:
    bar = "█" * int(imp * 60)
    print(f"    {name:<20}: {bar} {imp:.4f}")

## MODELING: Gradient Boosted Trees

In [0]:
# ── Gradient Boosted Trees — tidak perlu scaling ──
print("Training Gradient Boosted Trees...")
gbt = GBTRegressor(
    featuresCol="features",      # langsung dari VectorAssembler
    labelCol=TARGET_COL,
    maxIter=50,
    maxDepth=8,
    stepSize=0.1,
    subsamplingRate=0.8,
    minInstancesPerNode=5,
    seed=42
)
gbt_model  = gbt.fit(train_df)
gbt_preds  = gbt_model.transform(test_df)
gbt_result = evaluate(gbt_preds, "Gradient Boosted Trees (50 iter, depth 8)")

print("\n  Feature Importances (GBT):")
gbt_importances = sorted(
    zip(FEATURE_COLS, gbt_model.featureImportances),
    key=lambda x: x[1], reverse=True
)
for name, imp in gbt_importances:
    bar = "█" * int(imp * 60)
    print(f"    {name:<20}: {bar} {imp:.4f}")